**Repository edition:** The training code and all model/data fingerprints are unchanged. To keep dataset captions and saved predictions out of Git, this notebook reads the original `portable_models.json` from Drive instead of embedding it. Put that private sidecar at `/content/drive/MyDrive/portable_models.json` (or change `PORTABLE_MODELS_OVERRIDE`). It is required for both control modes; `TRAIN_CONTROL=True` only removes the need for `factorial_weights.zip`. Contracts are regenerated from the original train ZIP.

# SigLIP E0 · 최소 변경 오답으로 학습하기

**이 노트북 하나를 Colab에 올리고 무료 T4에서 위부터 실행하세요. 별도 코드 ZIP은 필요 없습니다.**

원본 ZIP 경로는 `/content/drive/MyDrive/inthon2026_train.zip`입니다. 기존 계정의 특징 캐시나
실행 기록은 필요하지 않습니다. 기본 설정은 옮겨 둔 `factorial_weights.zip`의 E0 가중치 3개를
기준 모델로 사용합니다. 기준 모델도 다시 학습하려면 아래 `TRAIN_CONTROL=True`로 바꾸세요.

이번 실험은 E0 구조를 유지하고 **학습용 3,336개에서 만든 최소 변경 오답**의 효과를 비교합니다.

- 기준: 무작위 원문 2개 + 팀 lexical 1개 + 유사 영상 원문 1개.
- 새 학습: 무작위 원문 2개 + 최소 변경 1개 + 유사 영상 원문 1개.
- 최소 변경은 색상·모양·명사·행동·상태·공간 관계 중 적용 가능한 유형을 균형 있게 선택합니다.
- 해당 영상에 유효한 최소 변경이 없거나 다른 선지와 충돌하면 기존 lexical 선지를 유지하고 횟수를 기록합니다.
- 검증 영상 834개와 그 영상에서 생성한 선지는 학습에 사용하지 않습니다.
- 새 모델은 기존 가중치에서 이어 학습하지 않고, 같은 seed의 초기 상태에서 3번 학습합니다.
- 기존 무작위·팀 검증과 최소 변경 검증을 고정합니다. 최소 변경 검증은 최종 비교용이며
  매 epoch 체크포인트를 고르는 데 사용하지 않습니다.
- Qwen 사건 추가 학습은 이번 실험에 포함되지 않습니다. 별도 품질 개선 후 비교할 단계입니다.

기본 설정은 기존 E0를 재사용하므로 새 학습 3회입니다. `TRAIN_CONTROL=True`면 동일한 새 특징으로
기준 3회 + 최소 변경 3회, 총 6회 학습합니다. 후자가 캐시 재추출에 따른 수치 차이도 함께 통제합니다.
두 경우 모두 제로샷을 함께 평가하고, 3 seed 평균은 앙상블 성능과 구분합니다.

원문을 규칙으로 바꿨다는 사실만으로 오답의 거짓이 보장되지는 않습니다. 생성 규칙·원문·변경 위치·
학습 스케줄을 함께 저장합니다. 이 점수는 개발 검증 결과이며 리더보드 점수가 아닙니다.


In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys
from google.colab import drive

TRAIN_ZIP_OVERRIDE = "/content/drive/MyDrive/inthon2026_train.zip"
PORTABLE_MODELS_OVERRIDE = "/content/drive/MyDrive/portable_models.json"
RUN_NAME = "siglip-minimal-training-v1"
WEIGHTS_DIR = Path("/content/drive/MyDrive/factorial_weights")
WEIGHTS_ZIP = Path("/content/drive/MyDrive/factorial_weights.zip")
TRAIN_CONTROL = False       # True: 기준 모델도 동일한 새 특징으로 3 seed 재학습
SMOKE = False               # True: head 1 epoch·seed 2026 점검. 원본 특징은 전체 추출
EPOCHS = 30
TRAIN_BATCH_SIZE = 32
TEXT_BATCH_SIZE = 16

if SMOKE:
    RUN_NAME += '-smoke'
if TRAIN_CONTROL:
    RUN_NAME += '-paired-control'
if not RUN_NAME or any(c not in 'abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789-_' for c in RUN_NAME):
    raise ValueError('RUN_NAME에는 영문·숫자·밑줄·하이픈을 사용하세요.')
if min(EPOCHS, TRAIN_BATCH_SIZE, TEXT_BATCH_SIZE) < 1:
    raise ValueError('epoch과 batch size는 1 이상이어야 합니다.')

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/Inthon_SigLIP_2026')
DRIVE_RUN = DRIVE_ROOT / 'runs' / RUN_NAME
LOCAL_KIT = Path('/content/inthon-siglip-minimal-training')
CONTRACT_DIR = DRIVE_ROOT / 'contracts/starter80-20-seed2026-v1'
TRAIN_ZIP = Path(TRAIN_ZIP_OVERRIDE)
if not TRAIN_ZIP.is_file():
    raise FileNotFoundError('원본 데이터 ZIP이 없습니다: ' + str(TRAIN_ZIP))
if not TRAIN_CONTROL and not WEIGHTS_ZIP.is_file():
    missing = [str(WEIGHTS_DIR / 'E0' / f'seed{seed}/best.pt') for seed in [2026,2027,2028]
               if not (WEIGHTS_DIR / 'E0' / f'seed{seed}/best.pt').is_file()]
    if missing:
        raise FileNotFoundError('기준 E0 가중치가 필요합니다. factorial_weights.zip을 올리거나 TRAIN_CONTROL=True로 설정하세요.\n' + '\n'.join(missing))
if not shutil.which('nvidia-smi'):
    raise RuntimeError('Colab 런타임 유형을 무료 T4 GPU로 바꿔주세요.')
gpu = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], text=True)
if len(gpu.strip().splitlines()) != 1 or 'T4' not in gpu:
    raise RuntimeError('이 노트북은 무료 T4 GPU 한 대에서 실행하세요: ' + gpu)
print(gpu)
DRIVE_RUN.mkdir(parents=True, exist_ok=True)
LOCAL_KIT.mkdir(parents=True, exist_ok=True)
print('결과 저장:', DRIVE_RUN)
print('기준 모델:', '같은 특징으로 재학습' if TRAIN_CONTROL else '기존 E0 3 seed 재사용')
if SMOKE:
    print('head 실행 점검 모드입니다. 전체 원본 특징은 추출하며 학습은 1 epoch·seed 2026만 실행합니다.')


## 내장 코드 준비
학습·검증·증강 코드를 노트북에서 꺼내고 무결성을 검사합니다.


In [ ]:
import base64, zlib
SOURCES = json.loads(zlib.decompress(base64.b64decode('...')).decode('utf-8'))
EXPECTED_HASHES = {'baseline_model.py': '613f72959d492e292d3f6fd28ff4e198e21039ea691cafdff1209f4627279389', 'common.py': '71fd0b64ce052d21bb53d748d96c7d483af28a7a08dbd92182adbf9f21d1d362', 'comparison_protocol.py': 'ff675ab96580c03c1626566beb071e72c2fdf4af1a6eb26c6509f5612e51952a', 'data_contract.py': '2427f7bd9909b937d991752cd48c927d0681aebd4963b7df44113deb5b853b72', 'dataset.py': '0b6057bc01bce70747e28de3745210127d424b933c1d96b1b570732468abc71a', 'download_train.py': 'b45803b4514fc2062296a85aff6391e1954ff80c92d311d0766cba50ed6ae15c', 'experiment_io.py': '8455ec7d76e2245f9c767f02b4b4f15ced45c4e36a9c6807e9ea7167f8470a16', 'factorial_data.py': '3dc205404ec473147a251141160e29b99d37629ab4aab2a69f810eaf0202e023', 'factorial_delivery.py': 'b24b89baff21f1ba4136624dfb4d9df3ccd7888a05b8484e09e82862c321a14d', 'factorial_models.py': 'e303ac3102c61a39256e58db04ce7e0984b4c0ab340b615ce6046a6ec6f37306', 'hard_negative_data.py': '25092c1480742c7307e5f28c31e30925c7b0f8190146f7e465c1f98cac49ec3f', 'minimal_training_data.py': 'b4a31a930c9741a1d6e1e24bcfdba7d6ca82328ad632504864874d3ad4c0f5c9', 'prepare_features.py': '9eba29c74cb5f786e43c9aec6be27ad3205ef6af5c4b38fe1e204915f9e55178', 'prepare_fresh_eval.py': 'f00a7cf7f7a5ed9f9a7ef15b09bd7ac3e05f34db375497ec986641a699cd5927', 'prepare_team_validation.py': 'e859ddd1e065f0cc10ca993c3605047d60b0939da95a0bdd0f02bcdf34940617', 'qwen_generate_candidates.py': '1019f3e8b60cc1e8544923fa2ba04f1853652c55a8ecd021dc722bdc6f7aeec8', 'review_candidates.py': 'b7bf745f1cf3cd8b9c31ba57bd87dad7d2a71ff40a86138876db79a3c831bf53', 'run_comparison.py': '58158d19ec306f8ee0e69b53df7091e49b16d804c36078109ec2f5643e1bddd6', 'run_factorial_comparison.py': '8754375d5d19ef6a9d1b271231fb6b5bd495881fd8b1e4ec6ff1efd545b0507b', 'run_hard_evaluation.py': '4c68ca110322f1e7ddc6ee5f6a1f1a1cee551f43090e5b65e6259143c7a89276', 'run_minimal_training.py': '605784ceb7eb65ea58e6983d9c7aeca6f88bd100ddf54e62384149369b04f75f', 'run_sentence_comparison.py': '22b0ef3809bc8310a91ef4188ff87ae78530f4c8a01196a253b9a4447d135b71', 'sentence_features.py': '80badb548234c6c1dd9b28478825d73b6cd0b247cf19430e83bb0260667493ad', 'sentence_models.py': '6a9e4d9311dee5b64779cd6ad0e6cfc79dfec7d5761808ae9eda16ae3c8a297d', 'siglip_model.py': '5011c6fb39e2f24d24e022932bfcd853def27eb3920cf68f8a751d18dd61fc61', 'team/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'team/lexical.py': '081cd1b2f588309f0c9054c7fedc3f46a1c0a44c83089c44feea76f5a6048f8f', 'team/negatives.py': '130dcd283eaef9b2e29fc8141e65aecc5f8d65b14439d138ccaf89fe57a66b74', 'team/train_negatives.py': 'd5883004bf945e2ba9a12f398f7920674b68773ba75367585c83e96a260a96d0', 'team_validation.py': '3bcd3429c6d89688b2bea3e5798b072a79ba80e55a92e430e770e6bbd90dfedf', 'temporal_models.py': '65bd5b5648d7508d689ea0670a83543af567f5cd53684d69dfadd860f66fbba3', 'tests/test_training_data.py': '89bfa2eca92061a4169d12a6b1f8be524d7a957b9c7c6c9a8c9de25516f32d2e', 'tests/test_training_features.py': 'd0e21946fd3157c8353c45d1801511bac62993b21078be6a84223c140df9dbf4', 'tests/test_training_runner.py': '47db8475a24d4c7e38d22ee98c654ab32dbbe5ebe2c3110bb107609c99140128', 'training_features.py': '9a858eac013063cf47200ecf131307c840de5f4b04cf484d569337646a598159', 'training_schedules.py': '4aeb0ced8bf5cb65cac5bd9d82a04b95e7a3c52a5d8d30f2107a5b17376d4690'}
for relative, source in SOURCES.items():
    if hashlib.sha256(source.encode()).hexdigest() != EXPECTED_HASHES[relative]:
        raise RuntimeError('내장 코드가 손상됐습니다: ' + relative)
    for root in [LOCAL_KIT, DRIVE_RUN / 'source']:
        path = root / relative
        if root != LOCAL_KIT and path.exists() and path.read_text(encoding='utf-8') != source:
            raise RuntimeError('기존 실행 코드와 다릅니다. RUN_NAME을 새 이름으로 바꿔주세요.')
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(source, encoding='utf-8', newline='\n')
print('내장 코드 준비 완료')


## 기준 E0 가중치 확인
기존 ZIP에 E1~E3가 들어 있어도 E0의 seed 2026·2027·2028만 읽습니다.


In [ ]:
PORTABLE_MODELS = LOCAL_KIT / 'portable_models.json'
portable_source = Path(PORTABLE_MODELS_OVERRIDE)
if not portable_source.is_file():
    raise FileNotFoundError('Place the original private portable_models.json at PORTABLE_MODELS_OVERRIDE; it contains validation data and is intentionally not in Git.')
portable_bytes = portable_source.read_bytes()
if hashlib.sha256(portable_bytes).hexdigest() != '57250ff11b7f487506f1ea497e54c32733a28ad4b784bbb11151ae9c0097e34c':
    raise RuntimeError('기존 모델 메타데이터가 손상됐습니다.')
PORTABLE_MODELS.write_bytes(portable_bytes)
portable = json.loads(portable_bytes)
e0_records = [r for r in portable['heads'] if r['experiment']=='E0']
if {r['seed'] for r in e0_records} != {2026,2027,2028} or len(e0_records) != 3:
    raise ValueError('기준 E0 seed 구성이 다릅니다.')
if not TRAIN_CONTROL:
    if WEIGHTS_ZIP.is_file():
        import zipfile
        WEIGHTS_DIR = Path('/content/inthon-minimal-e0-weights')
        WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(WEIGHTS_ZIP) as archive:
            if len(archive.namelist()) != len(set(archive.namelist())):
                raise ValueError('가중치 ZIP에 중복된 파일 이름이 있습니다.')
            for row in e0_records:
                name = f"E0/seed{row['seed']}/best.pt"
                record = row['checkpoint_record']
                info = archive.getinfo(name)
                if info.file_size != record['bytes']:
                    raise ValueError('기준 가중치 크기가 다릅니다: ' + name)
                data = archive.read(name)
                if hashlib.sha256(data).hexdigest() != record['sha256']:
                    raise ValueError('기준 가중치 해시가 다릅니다: ' + name)
                target = WEIGHTS_DIR / name
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_bytes(data)
    for row in e0_records:
        path = WEIGHTS_DIR / 'E0' / f"seed{row['seed']}/best.pt"
        if not path.is_file() or path.stat().st_size != row['checkpoint_record']['bytes'] or hashlib.sha256(path.read_bytes()).hexdigest() != row['checkpoint_record']['sha256']:
            raise ValueError('기준 E0 가중치가 기존 결과와 다릅니다: ' + str(path))
    print('기준 E0 가중치 3개 확인 완료')
else:
    print('기준 E0도 재학습합니다. 이전 가중치는 읽지 않습니다.')


## 같은 Python·패키지 환경 준비
기존 Colab에서 검증한 별도 Python 환경을 사용합니다. Colab 기본 패키지나 런타임 재시작에 의존하지 않습니다.
설치 후 환경 버전과 GPU를 확인하고 CPU 코드 검사를 실행합니다.


In [ ]:
PYTHON_VERSION = "3.12.3"
ENV_DIR = Path("/content/inthon-siglip-python312-cu128")
TOOL_DIR = Path("/content/inthon-colab-tools")
UV = TOOL_DIR / "bin/uv"
if not UV.is_file():
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--no-deps",
                    "--target", str(TOOL_DIR), "uv==0.13.4"], check=True)
if not (ENV_DIR / "bin/python").is_file():
    subprocess.run([str(UV), "python", "install", PYTHON_VERSION], check=True)
    subprocess.run([str(UV), "venv", "--python", PYTHON_VERSION,
                    "--seed", str(ENV_DIR)], check=True)
PYTHON = ENV_DIR / "bin/python"
subprocess.run([str(PYTHON), "-c",
                "import sys; assert sys.version_info[:3] == (3, 12, 3), sys.version"], check=True)

TORCH_PINS = ["torch==2.8.0+cu128", "torchvision==0.23.0+cu128", "torchaudio==2.8.0+cu128"]
PACKAGE_PINS = [
    "transformers==4.57.6", "numpy==2.5.4", "Pillow==12.3.0", "safetensors==0.8.0",
    "huggingface-hub==0.36.2", "tokenizers==0.22.2",
    "sentence-transformers==5.4.0", "einops==0.8.2", "timm==1.0.30",
    "scipy==1.18.1", "scikit-learn==1.9.1", "opencv-python-headless==4.14.0.94",
    "peft==0.18.1", "accelerate==1.15.0", "bitsandbytes==0.50.2",
    "cloudpickle==3.1.2", "joblib==1.6.0", "narwhals==2.28.0", "threadpoolctl==3.7.0",
]
PINS_PATH = DRIVE_RUN / "requirements-colab.txt"
PINS_PATH.write_text("\n".join(TORCH_PINS + PACKAGE_PINS) + "\n", encoding="utf-8")
subprocess.run([str(PYTHON), "-m", "pip", "install", "--no-cache-dir",
                "--index-url", "https://download.pytorch.org/whl/cu128", *TORCH_PINS], check=True)
subprocess.run([str(PYTHON), "-m", "pip", "install", "--no-cache-dir",
                "-r", str(PINS_PATH)], check=True)
subprocess.run([str(PYTHON), "-m", "pip", "check"], check=True)

ENV = os.environ.copy()
ENV.pop("PYTHONPATH", None)
ENV.pop("PYTHONHOME", None)
ENV.update({"HF_HOME": "/content/inthon-hf-cache", "HF_HUB_ENABLE_HF_TRANSFER": "0",
            "TOKENIZERS_PARALLELISM": "false", "PYTHONUNBUFFERED": "1",
            "PYTHONNOUSERSITE": "1", "OMP_NUM_THREADS": "2", "OPENBLAS_NUM_THREADS": "2"})
for name in ("HF_HUB_OFFLINE", "TRANSFORMERS_OFFLINE"):
    ENV.pop(name, None)

def run_command(args, *, log_name=None, extra_env=None, cwd=None):
    command = [str(arg) for arg in args]
    log = (DRIVE_RUN / log_name).open("a", encoding="utf-8") if log_name else None
    try:
        with subprocess.Popen(command, cwd=cwd or LOCAL_KIT,
                              env={**ENV, **(extra_env or {})}, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
            try:
                for line in process.stdout:
                    print(line, end="", flush=True)
                    if log:
                        log.write(line)
                        log.flush()
                returncode = process.wait()
            except BaseException:
                process.terminate()
                try:
                    process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()
                raise
        if returncode:
            raise subprocess.CalledProcessError(returncode, command)
    finally:
        if log:
            log.close()

def run_python(source, *args, **kwargs):
    run_command([PYTHON, "-u", "-c", source, *args], **kwargs)

def run_script(script, *args, **kwargs):
    run_command([PYTHON, "-u", LOCAL_KIT / script, *args], **kwargs)

freeze = subprocess.check_output([str(PYTHON), "-m", "pip", "freeze", "--all"], text=True, env=ENV)
(DRIVE_RUN / "requirements-installed.txt").write_text(freeze, encoding="utf-8")
print("실행 환경 준비 완료:", PYTHON)
ENV["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"


In [ ]:
run_python(r"""
import importlib.metadata as metadata, json, platform, sys
from pathlib import Path
import torch
from transformers import SiglipModel, PreTrainedTokenizerFast, AutoImageProcessor
assert torch.cuda.is_available(), "CUDA를 사용할 수 없습니다. T4 GPU 런타임을 확인하세요."
assert torch.version.cuda == "12.8", torch.version.cuda
versions = {}
for pin in Path(sys.argv[1]).read_text().splitlines():
    package, expected = pin.split("==")
    versions[package] = metadata.version(package)
    assert versions[package] == expected, (package, versions[package], expected)
with torch.inference_mode():
    x = torch.ones((32, 32), device="cuda", dtype=torch.float16)
    assert torch.isfinite(x @ x.T).all()
torch.cuda.synchronize()
report = {"python": platform.python_version(), "packages": versions,
          "gpu": torch.cuda.get_device_name(0), "cuda": torch.version.cuda,
          "vram_gib": torch.cuda.get_device_properties(0).total_memory / 2**30}
Path(sys.argv[2]).write_text(json.dumps(report, ensure_ascii=False, indent=2) + "\n")
print(json.dumps(report, indent=2))
""", PINS_PATH, DRIVE_RUN / "environment.json", log_name="environment.log")
run_command([PYTHON, "-m", "unittest", "discover", "-s", "tests", "-v"], log_name="code-tests.log")


## 기존 학습·검증 분할 복원
원본 ZIP을 Colab 임시 저장소로 복사하고 원문·분할·무작위 검증 선지의 해시를 확인합니다.
학습 3,336개와 검증 834개가 이전과 같은지 확인한 뒤 진행합니다.


In [ ]:
LOCAL_TRAIN_ZIP = Path('/content/inthon-data/inthon2026_train.zip')
LOCAL_TRAIN_ZIP.parent.mkdir(parents=True, exist_ok=True)
if not LOCAL_TRAIN_ZIP.is_file():
    temporary = LOCAL_TRAIN_ZIP.with_suffix('.zip.part')
    shutil.copyfile(TRAIN_ZIP, temporary)
    temporary.replace(LOCAL_TRAIN_ZIP)
TRAIN_ZIP = LOCAL_TRAIN_ZIP
run_script('data_contract.py', '--train-zip', TRAIN_ZIP, '--output-dir', CONTRACT_DIR, log_name='contracts.log')
EXPECTED_CONTRACT_HASHES = {'manifest.json': '6faf0c7fe6eb6d38798026e7ed83aecd628334d2c2d450560cde6da0fd224cd4', 'split.json': 'f341dbcce881ffefb11916bc523218f89aff00694f4b764f4b2da4009d98fb11', 'validation_candidates.json': '040b26dd0e8cd57029e5ab012b2423e907d85ddacae354951ef0abb59afa69be'}
for name, expected in EXPECTED_CONTRACT_HASHES.items():
    value = json.loads((CONTRACT_DIR / name).read_text(encoding="utf-8"))
    canonical = (json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":")) + "\n").encode()
    assert hashlib.sha256(canonical).hexdigest() == expected, ("계약 내용이 다릅니다", name)
split = json.loads((CONTRACT_DIR / "split.json").read_text())
assert len(split["train_ids"]) == 3336 and len(split["val_ids"]) == 834
print("고정 계약 일치: train=3336 / validation=834 / seed=2026 / negative seed=2027")


## E0 구조와 학습 범위

```text
전체 프레임 → 동결 SigLIP → 프레임별 특징 ──────┐
                                              ├→ 문장별 대응·시간 위치
선지 전체 → 문장·긴 문장 토큰 분할 → 동결 SigLIP ┘         ↓
                                              작은 문장 Transformer
                                                       ↓
기존 제로샷 코사인 점수 ──────────────── + 0.1 × tanh(보정값)
```

이미지·텍스트 인코더를 고정하고 E0의 작은 문장 대응 모듈만 학습합니다.
기존 제로샷 경로는 64토큰 입력을 유지하고, 보정 경로는 전체 문장·프레임을 처리합니다.
정답 인덱스를 이용한 분류 손실을 사용합니다. 제로샷이나 다른 모델의 예측을 학습 정답으로 사용하지 않습니다.

최소 변경 검증 점수를 보고 최적 epoch를 선택하지 않습니다. 기존 team/random 개발 검증 기준으로
epoch 0을 포함해 체크포인트를 고른 뒤, 새 최소 변경 검증을 한 번 계산합니다.
학습 중단 후 다시 실행하면 검증된 특징 캐시와 완료된 seed를 재사용합니다.


## 학습용 선지 생성 → 3 seed 학습 → 같은 검증 평가

처음 실행하면 원본 전체의 프레임·텍스트 특징을 추출합니다. 생성 선지의 텍스트 특징만 추가하고,
인코더는 고정합니다. Drive에 생성 데이터와 캐시가 저장되므로 같은 설정으로 이어서 실행할 수 있습니다.
형식만 같은 JSON의 자료형 차이를 정규화하고, 실제 데이터·설정·코드 차이는 검사합니다.

이 셀에는 채점 사이트 제출 기능이 없습니다. 전체 T4 실행 시간과 학습 결과는 실행 뒤 확인합니다.


In [ ]:
arguments = [
    '--contracts', CONTRACT_DIR, '--train-zip', TRAIN_ZIP,
    '--portable-models', PORTABLE_MODELS, '--output', DRIVE_RUN,
    '--device', 'cuda', '--epochs', EPOCHS,
    '--batch-size', TRAIN_BATCH_SIZE, '--text-batch-size', TEXT_BATCH_SIZE,
]
if TRAIN_CONTROL:
    arguments += ['--train-control']
else:
    arguments += ['--weights-dir', WEIGHTS_DIR]
if SMOKE:
    arguments += ['--smoke']
run_script('run_minimal_training.py', *arguments, log_name='training.log')


## 결과 확인 및 공유
`minimal_training_results.zip`을 공유하세요. 비교 점수·3 seed 결과·생성 데이터·오답 선택 기록·출처를 포함합니다.
모델 가중치와 특징 캐시·원본 이미지는 제외합니다. 가중치는 Drive의 실행 폴더에 따로 보존됩니다.
기존 random/team 점수가 유지되는지와 최소 변경 유형별 개선을 함께 확인하세요.
아래 화면은 주요 지표 5개만 표시하며 전체 유형별 점수는 CSV에 남습니다.
선택 epoch가 0이면 제로샷 체크포인트가 선택된 것이고, 오답 대체 fallback은 그 영상에서 기존 lexical
선지를 유지했다는 뜻입니다. 두 경우를 구분해서 확인하세요.


In [ ]:
import csv, html
from IPython.display import display, HTML
csv_path = DRIVE_RUN / 'comparison.csv'
if csv_path.is_file():
    with csv_path.open(encoding='utf-8-sig', newline='') as stream:
        all_rows = list(csv.DictReader(stream))
    selected_metrics = ['random_mean_rank_reward', 'team_mean_rank_reward', 'minimal_episode_win',
                        'minimal_5choice_reward', 'minimal_5choice_top1']
    filtered = [row for row in all_rows if row['metric'] in selected_metrics]
    rows = [['model', 'metric', 'mean', 'std']] + [[row['model'], row['metric'],
            f"{float(row['mean']):.6f}", f"{float(row['std']):.6f}"] for row in filtered]
    display(HTML('<table>' + ''.join('<tr>' + ''.join('<td style="padding:5px;border:1px solid #ddd">' + html.escape(v) + '</td>' for v in row) + '</tr>' for row in rows) + '</table>'))
report_path = DRIVE_RUN / 'comparison.json'
if report_path.is_file():
    report = json.loads(report_path.read_text(encoding='utf-8'))
    for model, data in report['models'].items():
        epochs = data.get('best_epochs', [])
        if any(epoch is not None for epoch in epochs):
            print(model, '선택 epoch:', epochs)
    for path in sorted((DRIVE_RUN / 'heads/minimal').glob('seed*/result.json')):
        result = json.loads(path.read_text(encoding='utf-8'))
        if result.get('baseline_fallback'):
            print('체크포인트 알림:', path.parent.name, 'epoch 0 제로샷이 선택됐습니다.')
    audits = []
    for path in sorted((DRIVE_RUN / 'negative_schedules/minimal').glob('seed*/epoch-*.json.gz.json')):
        audits.append(json.loads(path.read_text(encoding='utf-8'))['audit'])
    if audits:
        total = sum(row['episodes'] for row in audits)
        fallback = sum(row['fallback_episodes'] for row in audits)
        print(f'오답 대체 기록: 학습 스케줄 {total:,}회 중 {fallback:,}회({fallback / total:.1%})는 기존 lexical 선지를 유지했습니다.')
        print('fallback 이유·유형별 선택 기록:', DRIVE_RUN / 'negative_schedules/minimal')
for name in ['comparison.json', 'comparison.csv', 'status.json']:
    path = DRIVE_RUN / name
    if path.is_file(): print(name, path)
RESULT_ZIP = DRIVE_RUN / 'minimal_training_results.zip'
if not RESULT_ZIP.is_file():
    raise FileNotFoundError('완료된 결과 ZIP이 없습니다. 위 학습 로그를 확인하세요.')
print('공유할 파일:', RESULT_ZIP)
if SMOKE:
    print('이 파일은 실행 점검 결과입니다. 전체 실험은 SMOKE=False로 별도 실행하세요.')
# 브라우저로 다운로드하려면 아래 두 줄의 주석을 해제하세요.
# from google.colab import files
# files.download(str(RESULT_ZIP))
